# Folium ARCGIS

In [ ]:

# ============================================================
# STUDENT POPULATION DISTRIBUTION MAP
# 7,633 RECORDS
#
# LOCATION: Current residence (city / ZIP)
# SHAPE: Role
# COLOR: Permanent region
# SIZE: Number of records
#
# GEOCODING: ArcGIS
# VISUALIZATION: Folium
# ============================================================

import pandas as pd
import folium
import math
import html
import webbrowser
import os

from pathlib import Path
from tqdm.auto import tqdm

from geopy.geocoders import ArcGIS
from geopy.extra.rate_limiter import RateLimiter


DATA_PATH = Path(r"C:\Users\MARE\OneDrive\Documents\data311_project2\data\processed\population_addresses_cleaned_processed.csv"
)

OUTPUT_DIR = DATA_PATH.parent

CACHE_PATH = OUTPUT_DIR / "current_location_cache.csv"

OUTPUT_MAP = OUTPUT_DIR / "student_population_role_region_map.html"

SUMMARY_PATH = OUTPUT_DIR / "population_location_summary.csv"


df = pd.read_csv(
    DATA_PATH,
    dtype={
        "current_zip": "string"
    }
)

print(f"Loaded records: {len(df):,}")


# Validate columns

required_columns = [
    "pop_id",
    "role",
    "current_city",
    "current_state",
    "current_zip",
    "permanent_region"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")


#clean data

def clean_text(series):

    return (
        series.astype("string")
        .str.strip()
        .replace("", pd.NA)
    )


for col in required_columns[1:]:

    if col != "pop_id":
        df[col] = clean_text(df[col])


# Standardize role names

df["role"] = df["role"].str.strip().str.title()

# Standardize region names

df["permanent_region"] = (
    df["permanent_region"]
    .str.replace(r"\s+", "", regex=True)
    .str.strip()
)


# Standardize states

df["current_state"] = df["current_state"].str.upper()


# Standardize ZIP codes

df["current_zip"] = (
    df["current_zip"]
    .str.extract(r"(\d{1,5})", expand=False)
    .str.zfill(5)
)


# Known geographic correction

mck_mask = (
    df["current_city"]
    .str.contains("McKinleyville", case=False, na=False)
)

df.loc[mck_mask, "current_city"] = "McKinleyville"

df.loc[mck_mask, "current_state"] = "CA"


# standard roles

def standardize_role(role):

    role = str(role).strip().lower()

    if "faculty" in role:
        return "Faculty"

    if "student" in role:
        return "Student"

    if "staff" in role or "admin" in role:
        return "Staff/Admin"

    return "Other"


df["map_role"] = df["role"].apply(standardize_role)


# standard geo loc

region_mapping = {

    "NorCal": "NorCal",

    "SanDiego": "SanDiego",

    "LA": "LA",

    "OutOfState": "OutOfState",

    "BayArea": "BayArea"

}

df["map_region"] = df["permanent_region"].map(
    region_mapping
).fillna("Other")


# construct current loc

# Use city + state + ZIP when available.
# If city is missing, use ZIP + state.

df["location_query"] = (
    df["current_city"].fillna("")
    + ", "
    + df["current_state"].fillna("")
    + " "
    + df["current_zip"].fillna("")
).str.strip(" ,")


# Fallback for missing cities

missing_city = df["current_city"].isna()

df.loc[missing_city, "location_query"] = (
    df.loc[missing_city, "current_zip"].fillna("")
    + ", "
    + df.loc[missing_city, "current_state"].fillna("")
    + ", USA"
)


# Remove records without sufficient location information

df = df[
    df["current_zip"].str.fullmatch(r"\d{5}", na=False)
    & df["current_state"].notna()
].copy()


print(f"Records with valid ZIP formatting: {len(df):,}")


# agg pop counts

group_columns = [

    "current_city",

    "current_state",

    "current_zip",

    "location_query",

    "map_role",

    "map_region"

]


population = (

    df.groupby(
        group_columns,
        dropna=False
    )

    .agg(
        population_count=("pop_id", "nunique")
    )

    .reset_index()

)


print("\nAGGREGATION RESULTS")
print("-" * 50)

print(f"Original records: {len(df):,}")

print(f"Aggregated markers: {len(population):,}")

print(
    f"Records represented: "
    f"{population['population_count'].sum():,}"
)


# ID unique loc for geo coding

locations = (

    population[
        [
            "location_query",
            "current_city",
            "current_state",
            "current_zip"
        ]
    ]

    .drop_duplicates("location_query")

    .reset_index(drop=True)

)


print(
    f"\nUnique current locations: {len(locations):,}"
)


# initialize geo encoder arcgis

geolocator = ArcGIS(timeout=15)

geocode = RateLimiter(

    geolocator.geocode,

    min_delay_seconds=0.2,

    max_retries=2,

    error_wait_seconds=5,

    swallow_exceptions=True,

    return_value_on_exception=None

)



# load geocode caceh

if CACHE_PATH.exists():

    cache = pd.read_csv(CACHE_PATH)

else:

    cache = pd.DataFrame(
        columns=[
            "location_query",
            "latitude",
            "longitude"
        ]
    )


cache = cache.dropna(
    subset=["latitude", "longitude"]
)

cache = cache.drop_duplicates(
    subset=["location_query"],
    keep="last"
)


cached_locations = set(
    cache["location_query"]
)


new_locations = [

    location

    for location in locations["location_query"]

    if location not in cached_locations

]


# geocode unique loc..

new_results = []

print("\nGEOCODING CURRENT LOCATIONS")


for i, address in enumerate(

    tqdm(
        new_locations,
        desc="Geocoding"
    ),

    start=1

):

    try:

        result = geocode(address)

        if result:

            new_results.append({

                "location_query": address,

                "latitude": result.latitude,

                "longitude": result.longitude

            })


    except Exception as error:

        print(f"\nError: {address} | {error}")


    # Save every 25 locations

    if i % 25 == 0 and new_results:

        cache = pd.concat(

            [
                cache,
                pd.DataFrame(new_results)
            ],

            ignore_index=True

        )

        cache = cache.drop_duplicates(
            subset=["location_query"],
            keep="last"
        )

        cache.to_csv(
            CACHE_PATH,
            index=False
        )

        new_results = []


# Save remaining results

if new_results:

    cache = pd.concat(

        [
            cache,
            pd.DataFrame(new_results)
        ],

        ignore_index=True

    )


cache.to_csv(
    CACHE_PATH,
    index=False
)


# merge coordinate

population = population.merge(

    cache[
        [
            "location_query",
            "latitude",
            "longitude"
        ]
    ],

    on="location_query",

    how="left"

)


# Identify failed geocoding records

unmatched = population[
    population["latitude"].isna()
    |
    population["longitude"].isna()
].copy()


print(
    f"\nUnmatched aggregated markers: {len(unmatched):,}"
)


# Remove unmatched locations from map

population = population.dropna(
    subset=["latitude", "longitude"]
).copy()


# marker color

region_colors = {

    "NorCal": "green",

    "SanDiego": "blue",

    "LA": "red",

    "OutOfState": "purple",

    "BayArea": "orange",

    "Other": "gray"

}


# marker shape

role_icons = {

    "Student": "circle",

    "Staff/Admin": "square-full",

    "Faculty": "triangle"

}


# map detail

m = folium.Map(

    location=[40.8665, -124.0828],

    zoom_start=7,

    tiles="CartoDB positron",

    control_scale=True

)


# marker cluster

from folium.plugins import MarkerCluster

marker_cluster = MarkerCluster(

    name="Population Distribution",

    disableClusteringAtZoom=12

).add_to(m)


# add agg. markers

for row in tqdm(

    population.itertuples(index=False),

    total=len(population),

    desc="Building map"

):

    color = region_colors.get(
        row.map_region,
        "gray"
    )


    shape = role_icons.get(
        row.map_role,
        "circle"
    )


    # Scale markers based on population count

    radius = min(

        18,

        max(

            5,

            4 + 3 * math.log1p(
                row.population_count
            )

        )

    )


    # Popup without personally identifying information

    popup_html = f"""

    <div style="font-size:13px; width:200px">

        <b>Role:</b> {html.escape(row.map_role)}<br>

        <b>Permanent Region:</b>
        {html.escape(row.map_region)}<br>

        <b>Current Location:</b>
        {html.escape(str(row.current_city))}<br>

        <b>ZIP:</b>
        {html.escape(str(row.current_zip))}<br>

        <b>Population Count:</b>
        {row.population_count:,}

    </div>

    """


    folium.Marker(

        location=[
            row.latitude,
            row.longitude
        ],

        tooltip=(

            f"{row.map_role} | "

            f"{row.map_region} | "

            f"Count: {row.population_count:,}"

        ),

        popup=folium.Popup(
            popup_html,
            max_width=250
        ),

        icon=folium.DivIcon(

            html=f"""

            <div style="

                width:{radius * 2}px;

                height:{radius * 2}px;

                background-color:{color};

                border:2px solid white;

                border-radius:{
                    '50%' if shape == 'circle'
                    else '0' if shape == 'square-full'
                    else '0'
                };

                clip-path:{
                    'polygon(50% 0%, 0% 100%, 100% 100%)'
                    if shape == 'triangle'
                    else 'none'
                };

                opacity:0.85;

                box-shadow:0 1px 3px #555;

            "></div>

            """,

            icon_size=(
                radius * 2,
                radius * 2
            ),

            icon_anchor=(
                radius,
                radius
            )

        )

    ).add_to(marker_cluster)


# create legend

legend_html = """

<div style="

position: fixed;

bottom: 30px;

left: 30px;

width: 245px;

background-color: white;

border: 1px solid #ccc;

border-radius: 6px;

padding: 12px;

z-index: 9999;

font-size: 12px;

box-shadow: 0 2px 6px rgba(0,0,0,0.2);

">

<b>POPULATION DISTRIBUTION</b>

<hr>

<b>ROLE — SHAPE</b><br>

● Student<br>

■ Staff / Admin<br>

▲ Faculty<br>

<hr>

<b>PERMANENT REGION — COLOR</b><br>

<span style="color:green">●</span> NorCal<br>

<span style="color:blue">●</span> San Diego<br>

<span style="color:red">●</span> LA<br>

<span style="color:purple">●</span> OutOfState<br>

<span style="color:orange">●</span> BayArea<br>

<hr>

<b>MARKER SIZE</b><br>

Larger marker = More records

</div>

"""

m.get_root().html.add_child(
    folium.Element(legend_html)
)


# map 

m.save(OUTPUT_MAP)


# export summeyr

summary = (

    population

    .groupby(
        [
            "map_role",
            "map_region"
        ],

        as_index=False
    )

    .agg(
        population_count=(
            "population_count",
            "sum"
        )
    )

)


summary.to_csv(
    SUMMARY_PATH,
    index=False
)


# final output

print("\n")
print("=" * 60)

print("MAP SUCCESSFULLY CREATED")

print("=" * 60)

print(f"Original records: {len(df):,}")

print(
    f"Aggregated markers: {len(population):,}"
)

print(
    f"Records represented: "
    f"{population['population_count'].sum():,}"
)

print(
    f"Unmatched markers: {len(unmatched):,}"
)

print("\nOUTPUT FILES")

print(f"Map: {OUTPUT_MAP}")

print(f"Cache: {CACHE_PATH}")

print(f"Summary: {SUMMARY_PATH}")

print("=" * 60)


Loaded records: 7,633
Records with valid ZIP formatting: 7,633

AGGREGATION RESULTS
--------------------------------------------------
Original records: 7,633
Aggregated markers: 42
Records represented: 7,633

Unique current locations: 6

GEOCODING CURRENT LOCATIONS


Geocoding: 0it [00:00, ?it/s]


Unmatched aggregated markers: 0


c:\Users\MARE\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\folium\raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


Building map:   0%|          | 0/42 [00:00<?, ?it/s]



MAP SUCCESSFULLY CREATED
Original records: 7,633
Aggregated markers: 42
Records represented: 7,633
Unmatched markers: 0

OUTPUT FILES
Map: C:\Users\MARE\OneDrive\Documents\data311_project2\data\processed\student_population_role_region_map.html
Cache: C:\Users\MARE\OneDrive\Documents\data311_project2\data\processed\current_location_cache.csv
Summary: C:\Users\MARE\OneDrive\Documents\data311_project2\data\processed\population_location_summary.csv


# Arc GIS OPENMAPS

In [26]:

import os
import re
import math
import html
import webbrowser
import pandas as pd
import folium

from tqdm import tqdm
from geopy.geocoders import ArcGIS
from geopy.extra.rate_limiter import RateLimiter
from folium.plugins import MarkerCluster


# file config

INPUT_FILE = (
    r"C:\Users\MARE\OneDrive\Documents\data311_project2\data\processed\population_addresses_cleaned_processed.csv"
)

OUTPUT_DIR = (
    r"C:\Users\MARE\OneDrive\Documents\data311_project2"
    r"\data\processed"
)

MAP_FILE = "student_population_role_region_map.html"
CACHE_FILE = "current_location_cache.csv"
SUMMARY_FILE = "population_location_summary.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_MAP = os.path.join(OUTPUT_DIR, MAP_FILE)
CACHE_PATH = os.path.join(OUTPUT_DIR, CACHE_FILE)
SUMMARY_PATH = os.path.join(OUTPUT_DIR, SUMMARY_FILE)


# map config

MAP_CENTER = [40.80, -124.10]
ZOOM_START = 8

# ZIP-level aggregation is recommended to reduce clutter.
# Available options: "zip" or "address".
LOCATION_LEVEL = "zip"

ROLE_SHAPES = {
    "Student": "circle",
    "Staff/Admin": "square",
    "Faculty": "triangle"
}

REGION_COLORS = {
    "NorCal": "#2E8B57",
    "SanDiego": "#2878B5",
    "LA": "#D62728",
    "OutOfState": "#8A2BE2",
    "BayArea": "#F28E2B",
    "Unknown": "#808080"
}


# data

print("\n" + "=" * 65)
print("LOADING POPULATION DATA")
print("=" * 65)

df = pd.read_csv(
    INPUT_FILE,
    dtype={
        "current_zip": "string",
        "current_state": "string",
        "current_city": "string",
        "current_street": "string",
        "permanent_region": "string",
        "role": "string"
    }
)

print(f"Loaded records: {len(df):,}")

required_columns = [
    "pop_id",
    "role",
    "current_street",
    "current_city",
    "current_state",
    "current_zip",
    "permanent_region"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# clean loc fields

def clean_text(series):
    return (
        series.astype("string")
        .str.strip()
        .replace("", pd.NA)
    )


for col in [
    "current_street",
    "current_city",
    "current_state",
    "current_zip",
    "role",
    "permanent_region"
]:
    df[col] = clean_text(df[col])


# Preserve valid five-digit ZIP codes, including leading zeros.
# ZIP+4 values are reduced to their first five digits.
zip_digits = (
    df["current_zip"]
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

df["current_zip"] = zip_digits.str.extract(
    r"^(\d{5})(?:-\d{4})?$",
    expand=False
)

valid_zip_count = df["current_zip"].notna().sum()

print(
    f"Records with valid ZIP formatting: "
    f"{valid_zip_count:,}"
)


# standard role

def standardize_role(value):

    if pd.isna(value):
        return "Other"

    value = str(value).strip().lower()

    if "student" in value:
        return "Student"

    if "faculty" in value or "professor" in value:
        return "Faculty"

    if (
        "staff" in value
        or "admin" in value
        or "administrator" in value
    ):
        return "Staff/Admin"

    return "Other"


df["map_role"] = df["role"].apply(
    standardize_role
)


# standard regions

def standardize_region(value):

    if pd.isna(value):
        return "Unknown"

    value = str(value).strip().lower()

    region_mapping = {
        "norcal": "NorCal",
        "northern california": "NorCal",
        "north california": "NorCal",

        "sandiego": "SanDiego",
        "san diego": "SanDiego",

        "la": "LA",
        "los angeles": "LA",

        "outofstate": "OutOfState",
        "out of state": "OutOfState",

        "bayarea": "BayArea",
        "bay area": "BayArea"
    }

    return region_mapping.get(value, "Unknown")


df["map_region"] = df["permanent_region"].apply(
    standardize_region
)


# geolocation queries

def create_location_query(row):

    street = row["current_street"]
    city = row["current_city"]
    state = row["current_state"]
    zipcode = row["current_zip"]

    if LOCATION_LEVEL == "address":

        parts = [
            street,
            city,
            state,
            zipcode,
            "USA"
        ]

    else:

        # ZIP-level mode is less precise but more readable.
        # If ZIP is missing, fall back to city/state.
        if pd.notna(zipcode):
            parts = [city, state, zipcode, "USA"]
        else:
            parts = [city, state, "USA"]

    parts = [
        str(part).strip()
        for part in parts
        if pd.notna(part) and str(part).strip()
    ]

    if len(parts) <= 1:
        return None

    return ", ".join(parts)


df["location_query"] = df.apply(
    create_location_query,
    axis=1
)

df = df[df["location_query"].notna()].copy()


# aggregate pop records

print("\n" + "=" * 65)
print("AGGREGATION RESULTS")
print("=" * 65)

aggregation_columns = [
    "location_query",
    "map_role",
    "map_region"
]

grouped = (
    df.groupby(
        aggregation_columns,
        dropna=False
    )
    .agg(
        population_count=("pop_id", "size"),
        current_city=("current_city", "first"),
        current_state=("current_state", "first"),
        current_zip=("current_zip", "first")
    )
    .reset_index()
)

print(f"Original records: {len(df):,}")
print(f"Aggregated markers: {len(grouped):,}")
print(
    f"Records represented: "
    f"{grouped['population_count'].sum():,}"
)
print(
    f"Unique current locations: "
    f"{grouped['location_query'].nunique():,}"
)


# geocoding cache

if os.path.exists(CACHE_PATH):

    cache_df = pd.read_csv(
        CACHE_PATH,
        dtype={"location_query": "string"}
    )

    print(
        f"\nLoaded geocoding cache: "
        f"{len(cache_df):,} locations"
    )

else:

    cache_df = pd.DataFrame(
        columns=[
            "location_query",
            "latitude",
            "longitude"
        ]
    )


cache = {}

for _, row in cache_df.iterrows():

    if (
        pd.notna(row["latitude"])
        and pd.notna(row["longitude"])
    ):

        cache[row["location_query"]] = (
            float(row["latitude"]),
            float(row["longitude"])
        )


# geocodingunique current locations

print("\n" + "=" * 65)
print("GEOCODING CURRENT LOCATIONS")
print("=" * 65)

geolocator = ArcGIS(
    timeout=15
)

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.0,
    max_retries=2,
    error_wait_seconds=5.0,
    swallow_exceptions=True
)

unique_locations = (
    grouped["location_query"]
    .dropna()
    .unique()
)

new_cache_records = []

for location in tqdm(
    unique_locations,
    desc="Geocoding"
):

    if location in cache:
        continue

    try:

        result = geocode(location)

        if result:

            latitude = result.latitude
            longitude = result.longitude

            cache[location] = (
                latitude,
                longitude
            )

            new_cache_records.append({
                "location_query": location,
                "latitude": latitude,
                "longitude": longitude
            })

    except Exception as error:

        print(
            f"\nGeocoding error for {location}: "
            f"{error}"
        )


# Save new geocoding results.
if new_cache_records:

    new_cache_df = pd.DataFrame(
        new_cache_records
    )

    cache_df = pd.concat(
        [cache_df, new_cache_df],
        ignore_index=True
    )

    cache_df = cache_df.drop_duplicates(
        subset=["location_query"],
        keep="last"
    )

    cache_df.to_csv(
        CACHE_PATH,
        index=False
    )


# attach coordinates to aggregated data

grouped["latitude"] = grouped[
    "location_query"
].map(
    lambda x: cache.get(x, (None, None))[0]
)

grouped["longitude"] = grouped[
    "location_query"
].map(
    lambda x: cache.get(x, (None, None))[1]
)

mapped = grouped.dropna(
    subset=["latitude", "longitude"]
).copy()

unmatched = grouped[
    grouped["latitude"].isna()
    | grouped["longitude"].isna()
].copy()

print(
    f"\nSuccessfully geocoded markers: "
    f"{len(mapped):,}"
)

print(
    f"Unmatched aggregated markers: "
    f"{len(unmatched):,}"
)


# folium map

print("\n" + "=" * 65)
print("BUILDING MAP")
print("=" * 65)

m = folium.Map(
    location=[40.80, -124.10],
    zoom_start=8,
    tiles="OpenStreetMap",
    control_scale=True
)

marker_cluster = MarkerCluster(
    name="Population Distribution",
    overlay=True,
    control=True,
    show=True,
    options={
        "maxClusterRadius": 35,
        "spiderfyOnMaxZoom": True,
        "showCoverageOnHover": False,
        "zoomToBoundsOnClick": True
    }
)

marker_cluster.add_to(m)

folium.TileLayer(
    tiles="https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors',
    name="OpenStreetMap",
    overlay=False,
    control=True
).add_to(m)

print("\nMAP COORDINATE VALIDATION")
print("-" * 50)

print("Mapped records:", len(mapped))
print("Missing latitude:", mapped["latitude"].isna().sum())
print("Missing longitude:", mapped["longitude"].isna().sum())

print("\nSample coordinates:")
print(
    mapped[
        ["location_query", "latitude", "longitude", "population_count"]
    ].head(10).to_string(index=False)
)

print("\nLatitude range:")
print(mapped["latitude"].min(), mapped["latitude"].max())

print("\nLongitude range:")
print(mapped["longitude"].min(), mapped["longitude"].max())

# role marker shape

def create_marker_html(role, region, count):

    shape = ROLE_SHAPES.get(
        role,
        "circle"
    )

    color = REGION_COLORS.get(
        region,
        "#808080"
    )

    # Marker size increases with population count.
    radius = min(
        18,
        max(
            6,
            5 + 2.5 * math.log1p(count)
        )
    )

    size = radius * 2

    common_style = (
        f"width:{size}px;"
        f"height:{size}px;"
        f"background-color:{color};"
        "border:2px solid white;"
        "box-shadow:0 1px 4px rgba(0,0,0,0.5);"
    )

    if shape == "circle":

        shape_style = (
            "border-radius:50%;"
        )

    elif shape == "square":

        shape_style = (
            "border-radius:2px;"
        )

    elif shape == "triangle":

        shape_style = (
            "clip-path:polygon(50% 0%, 0% 100%, 100% 100%);"
            "border:none;"
        )

    else:

        shape_style = (
            "border-radius:50%;"
        )

    return f"""
    <div style="
        {common_style}
        {shape_style}
    "></div>
    """


# aggregated markers

for _, row in tqdm(
    mapped.iterrows(),
    total=len(mapped),
    desc="Building map"
):

    role = row["map_role"]
    region = row["map_region"]
    count = int(row["population_count"])

    latitude = float(row["latitude"])
    longitude = float(row["longitude"])

    marker_html = create_marker_html(
        role,
        region,
        count
    )

    city = (
        html.escape(str(row["current_city"]))
        if pd.notna(row["current_city"])
        else "Unknown"
    )

    state = (
        html.escape(str(row["current_state"]))
        if pd.notna(row["current_state"])
        else "Unknown"
    )

    zipcode = (
        html.escape(str(row["current_zip"]))
        if pd.notna(row["current_zip"])
        else "Unknown"
    )

    popup_html = f"""
    <div style="font-family:Arial; width:220px;">

        <h4 style="margin-bottom:8px;">
            Population Location
        </h4>

        <b>Current City:</b> {city}<br>
        <b>State:</b> {state}<br>
        <b>ZIP Code:</b> {zipcode}<br>

        <hr>

        <b>Role:</b> {html.escape(role)}<br>
        <b>Permanent Region:</b> {html.escape(region)}<br>

        <hr>

        <b>Population Count:</b>
        <span style="font-size:18px; font-weight:bold;">
            {count:,}
        </span>

    </div>
    """

    icon = folium.DivIcon(
        html=marker_html,
        icon_size=(40, 40),
        icon_anchor=(20, 20)
    )

    folium.Marker(
        location=[
            latitude,
            longitude
        ],
        icon=icon,
        popup=folium.Popup(
            popup_html,
            max_width=300
        ),
        tooltip=(
            f"{role} | {region} | "
            f"Population: {count:,}"
        )
    ).add_to(marker_cluster)


# map legend

role_legend = """
<div style="
    position: fixed;
    bottom: 25px;
    left: 25px;
    width: 225px;
    background-color: white;
    border: 2px solid #555;
    border-radius: 7px;
    z-index: 9999;
    padding: 12px;
    font-family: Arial;
    font-size: 12px;
    box-shadow: 0 2px 6px rgba(0,0,0,0.3);
">

    <h4 style="margin-top:0;">
        Population Map Legend
    </h4>

    <b>Marker Shape = Role</b>

    <div style="margin-top:8px;">
        <span style="
            display:inline-block;
            width:12px;
            height:12px;
            border-radius:50%;
            background:#777;
            margin-right:8px;
        "></span>
        Student
    </div>

    <div style="margin-top:6px;">
        <span style="
            display:inline-block;
            width:12px;
            height:12px;
            background:#777;
            margin-right:8px;
        "></span>
        Staff/Admin
    </div>

    <div style="margin-top:6px;">
        <span style="
            display:inline-block;
            width:0;
            height:0;
            border-left:7px solid transparent;
            border-right:7px solid transparent;
            border-bottom:13px solid #777;
            margin-right:8px;
        "></span>
        Faculty
    </div>

    <hr>

    <b>Marker Color = Permanent Region</b>

    <div style="margin-top:8px;">
        <span style="color:#2E8B57;">●</span>
        NorCal
    </div>

    <div>
        <span style="color:#2878B5;">●</span>
        San Diego
    </div>

    <div>
        <span style="color:#D62728;">●</span>
        Los Angeles
    </div>

    <div>
        <span style="color:#8A2BE2;">●</span>
        Out of State
    </div>

    <div>
        <span style="color:#F28E2B;">●</span>
        Bay Area
    </div>

    <hr>

    <b>Marker Size = Population Count</b>

    <div style="font-size:11px; color:#555;">
        Larger markers represent more records.
    </div>

</div>
"""

m.get_root().html.add_child(
    folium.Element(role_legend)
)


# control layer

folium.LayerControl(
    collapsed=True
).add_to(m)


# pop usmmary

summary = (
    mapped.groupby(
        ["current_city", "current_state", "current_zip"],
        dropna=False
    )
    .agg(
        total_population=("population_count", "sum"),
        unique_role_region_combinations=(
            "population_count",
            "size"
        )
    )
    .reset_index()
    .sort_values(
        "total_population",
        ascending=False
    )
)

summary.to_csv(
    SUMMARY_PATH,
    index=False
)

folium.Marker(
    location=[40.8021, -124.1637],
    popup="Test Marker - Eureka, California",
    tooltip="Test Marker"
).add_to(m)

# validate

original_records = len(df)

records_represented = mapped[
    "population_count"
].sum()

aggregated_markers = len(grouped)

unmatched_markers = len(unmatched)

print("\n" + "=" * 65)
print("MAP SUCCESSFULLY CREATED")
print("=" * 65)

print(f"Original records: {original_records:,}")
print(f"Aggregated markers: {aggregated_markers:,}")
print(f"Mapped markers: {len(mapped):,}")
print(f"Records represented: {records_represented:,}")
print(f"Unmatched markers: {unmatched_markers:,}")

if records_represented + unmatched["population_count"].sum() != original_records:
    print(
        "\nWARNING: Some records were excluded "
        "or not represented in the aggregation."
    )

print("\nOUTPUT FILES")
print("-" * 65)

print(f"Map: {OUTPUT_MAP}")
print(f"Cache: {CACHE_PATH}")
print(f"Summary: {SUMMARY_PATH}")

print("=" * 65)


# save and open map

m.save(OUTPUT_MAP)

if os.path.exists(OUTPUT_MAP):

    print("\nMap saved successfully.")

    print(
        f"File size: "
        f"{os.path.getsize(OUTPUT_MAP) / 1024:.2f} KB"
    )

    print("\nOpening map in your default browser...")

    webbrowser.open(
        "file:///" + OUTPUT_MAP.replace("\\", "/")
    )

else:

    print("\nERROR: Map file was not created.")



LOADING POPULATION DATA
Loaded records: 7,633
Records with valid ZIP formatting: 7,633

AGGREGATION RESULTS
Original records: 7,633
Aggregated markers: 42
Records represented: 7,633
Unique current locations: 6

Loaded geocoding cache: 12 locations

GEOCODING CURRENT LOCATIONS


Geocoding: 100%|██████████| 6/6 [00:00<00:00, 34807.50it/s]



Successfully geocoded markers: 42
Unmatched aggregated markers: 0

BUILDING MAP

MAP COORDINATE VALIDATION
--------------------------------------------------
Mapped records: 42
Missing latitude: 0
Missing longitude: 0

Sample coordinates:
           location_query  latitude   longitude  population_count
   Arcata, CA, 95521, USA 40.875864 -124.084283               151
   Arcata, CA, 95521, USA 40.875864 -124.084283               147
   Arcata, CA, 95521, USA 40.875864 -124.084283               829
   Arcata, CA, 95521, USA 40.875864 -124.084283               922
   Arcata, CA, 95521, USA 40.875864 -124.084283               644
   Arcata, CA, 95521, USA 40.875864 -124.084283               866
   Arcata, CA, 95521, USA 40.875864 -124.084283               434
Blue Lake, CA, 95525, USA 40.879713 -123.979287                36
Blue Lake, CA, 95525, USA 40.879713 -123.979287                32
Blue Lake, CA, 95525, USA 40.879713 -123.979287                24

Latitude range:
40.5847485 41.067

Building map: 100%|██████████| 42/42 [00:00<00:00, 2786.25it/s]


MAP SUCCESSFULLY CREATED
Original records: 7,633
Aggregated markers: 42
Mapped markers: 42
Records represented: 7,633
Unmatched markers: 0

OUTPUT FILES
-----------------------------------------------------------------
Map: C:\Users\MARE\OneDrive\Documents\data311_project2\data\processed\student_population_role_region_map.html
Cache: C:\Users\MARE\OneDrive\Documents\data311_project2\data\processed\current_location_cache.csv
Summary: C:\Users\MARE\OneDrive\Documents\data311_project2\data\processed\population_location_summary.csv



Map saved successfully.
File size: 90.78 KB

Opening map in your default browser...


# Esri World Street Map

In [27]:

import os
import re
import math
import html
import webbrowser
import pandas as pd
import folium

from tqdm import tqdm
from geopy.geocoders import ArcGIS
from geopy.extra.rate_limiter import RateLimiter
from folium.plugins import MarkerCluster


# file config

INPUT_FILE = (
    r"C:\Users\MARE\OneDrive\Documents\data311_project2\data\processed\population_addresses_cleaned_processed.csv"
)

OUTPUT_DIR = (
    r"C:\Users\MARE\OneDrive\Documents\data311_project2"
    r"\data\processed"
)

MAP_FILE = "student_population_role_region_map.html"
CACHE_FILE = "current_location_cache.csv"
SUMMARY_FILE = "population_location_summary.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_MAP = os.path.join(OUTPUT_DIR, MAP_FILE)
CACHE_PATH = os.path.join(OUTPUT_DIR, CACHE_FILE)
SUMMARY_PATH = os.path.join(OUTPUT_DIR, SUMMARY_FILE)


# map config

MAP_CENTER = [40.80, -124.10]
ZOOM_START = 8

# ZIP-level aggregation is recommended to reduce clutter.
# Available options: "zip" or "address".
LOCATION_LEVEL = "zip"

ROLE_SHAPES = {
    "Student": "circle",
    "Staff/Admin": "square",
    "Faculty": "triangle"
}

REGION_COLORS = {
    "NorCal": "#2E8B57",
    "SanDiego": "#2878B5",
    "LA": "#D62728",
    "OutOfState": "#8A2BE2",
    "BayArea": "#F28E2B",
    "Unknown": "#808080"
}


# data

print("\n" + "=" * 65)
print("LOADING POPULATION DATA")
print("=" * 65)

df = pd.read_csv(
    INPUT_FILE,
    dtype={
        "current_zip": "string",
        "current_state": "string",
        "current_city": "string",
        "current_street": "string",
        "permanent_region": "string",
        "role": "string"
    }
)

print(f"Loaded records: {len(df):,}")

required_columns = [
    "pop_id",
    "role",
    "current_street",
    "current_city",
    "current_state",
    "current_zip",
    "permanent_region"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# clean loc fields

def clean_text(series):
    return (
        series.astype("string")
        .str.strip()
        .replace("", pd.NA)
    )


for col in [
    "current_street",
    "current_city",
    "current_state",
    "current_zip",
    "role",
    "permanent_region"
]:
    df[col] = clean_text(df[col])


# Preserve valid five-digit ZIP codes, including leading zeros.
# ZIP+4 values are reduced to their first five digits.
zip_digits = (
    df["current_zip"]
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

df["current_zip"] = zip_digits.str.extract(
    r"^(\d{5})(?:-\d{4})?$",
    expand=False
)

valid_zip_count = df["current_zip"].notna().sum()

print(
    f"Records with valid ZIP formatting: "
    f"{valid_zip_count:,}"
)


# standard role

def standardize_role(value):

    if pd.isna(value):
        return "Other"

    value = str(value).strip().lower()

    if "student" in value:
        return "Student"

    if "faculty" in value or "professor" in value:
        return "Faculty"

    if (
        "staff" in value
        or "admin" in value
        or "administrator" in value
    ):
        return "Staff/Admin"

    return "Other"


df["map_role"] = df["role"].apply(
    standardize_role
)


# standard regions

def standardize_region(value):

    if pd.isna(value):
        return "Unknown"

    value = str(value).strip().lower()

    region_mapping = {
        "norcal": "NorCal",
        "northern california": "NorCal",
        "north california": "NorCal",

        "sandiego": "SanDiego",
        "san diego": "SanDiego",

        "la": "LA",
        "los angeles": "LA",

        "outofstate": "OutOfState",
        "out of state": "OutOfState",

        "bayarea": "BayArea",
        "bay area": "BayArea"
    }

    return region_mapping.get(value, "Unknown")


df["map_region"] = df["permanent_region"].apply(
    standardize_region
)


# geolocation queries

def create_location_query(row):

    street = row["current_street"]
    city = row["current_city"]
    state = row["current_state"]
    zipcode = row["current_zip"]

    if LOCATION_LEVEL == "address":

        parts = [
            street,
            city,
            state,
            zipcode,
            "USA"
        ]

    else:

        # ZIP-level mode is less precise but more readable.
        # If ZIP is missing, fall back to city/state.
        if pd.notna(zipcode):
            parts = [city, state, zipcode, "USA"]
        else:
            parts = [city, state, "USA"]

    parts = [
        str(part).strip()
        for part in parts
        if pd.notna(part) and str(part).strip()
    ]

    if len(parts) <= 1:
        return None

    return ", ".join(parts)


df["location_query"] = df.apply(
    create_location_query,
    axis=1
)

df = df[df["location_query"].notna()].copy()


# aggregate pop records

print("\n" + "=" * 65)
print("AGGREGATION RESULTS")
print("=" * 65)

aggregation_columns = [
    "location_query",
    "map_role",
    "map_region"
]

grouped = (
    df.groupby(
        aggregation_columns,
        dropna=False
    )
    .agg(
        population_count=("pop_id", "size"),
        current_city=("current_city", "first"),
        current_state=("current_state", "first"),
        current_zip=("current_zip", "first")
    )
    .reset_index()
)

print(f"Original records: {len(df):,}")
print(f"Aggregated markers: {len(grouped):,}")
print(
    f"Records represented: "
    f"{grouped['population_count'].sum():,}"
)
print(
    f"Unique current locations: "
    f"{grouped['location_query'].nunique():,}"
)


# geocoding cache

if os.path.exists(CACHE_PATH):

    cache_df = pd.read_csv(
        CACHE_PATH,
        dtype={"location_query": "string"}
    )

    print(
        f"\nLoaded geocoding cache: "
        f"{len(cache_df):,} locations"
    )

else:

    cache_df = pd.DataFrame(
        columns=[
            "location_query",
            "latitude",
            "longitude"
        ]
    )


cache = {}

for _, row in cache_df.iterrows():

    if (
        pd.notna(row["latitude"])
        and pd.notna(row["longitude"])
    ):

        cache[row["location_query"]] = (
            float(row["latitude"]),
            float(row["longitude"])
        )


# geocodingunique current locations

print("\n" + "=" * 65)
print("GEOCODING CURRENT LOCATIONS")
print("=" * 65)

geolocator = ArcGIS(
    timeout=15
)

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.0,
    max_retries=2,
    error_wait_seconds=5.0,
    swallow_exceptions=True
)

unique_locations = (
    grouped["location_query"]
    .dropna()
    .unique()
)

new_cache_records = []

for location in tqdm(
    unique_locations,
    desc="Geocoding"
):

    if location in cache:
        continue

    try:

        result = geocode(location)

        if result:

            latitude = result.latitude
            longitude = result.longitude

            cache[location] = (
                latitude,
                longitude
            )

            new_cache_records.append({
                "location_query": location,
                "latitude": latitude,
                "longitude": longitude
            })

    except Exception as error:

        print(
            f"\nGeocoding error for {location}: "
            f"{error}"
        )


# Save new geocoding results.
if new_cache_records:

    new_cache_df = pd.DataFrame(
        new_cache_records
    )

    cache_df = pd.concat(
        [cache_df, new_cache_df],
        ignore_index=True
    )

    cache_df = cache_df.drop_duplicates(
        subset=["location_query"],
        keep="last"
    )

    cache_df.to_csv(
        CACHE_PATH,
        index=False
    )


# attach coordinates to aggregated data

grouped["latitude"] = grouped[
    "location_query"
].map(
    lambda x: cache.get(x, (None, None))[0]
)

grouped["longitude"] = grouped[
    "location_query"
].map(
    lambda x: cache.get(x, (None, None))[1]
)

mapped = grouped.dropna(
    subset=["latitude", "longitude"]
).copy()

unmatched = grouped[
    grouped["latitude"].isna()
    | grouped["longitude"].isna()
].copy()

print(
    f"\nSuccessfully geocoded markers: "
    f"{len(mapped):,}"
)

print(
    f"Unmatched aggregated markers: "
    f"{len(unmatched):,}"
)


# folium map

print("\n" + "=" * 65)
print("BUILDING MAP")
print("=" * 65)

m = folium.Map(
    location=[40.80, -124.10],
    zoom_start=9,
    tiles=None,
    control_scale=True
)

folium.TileLayer(
    tiles=(
        "https://server.arcgisonline.com/ArcGIS/rest/services/"
        "World_Street_Map/MapServer/tile/{z}/{y}/{x}"
    ),
    attr="Tiles © Esri",
    name="Esri World Street Map",
    overlay=False,
    control=True
).add_to(m)

marker_cluster = MarkerCluster(
    name="Population Distribution",
    overlay=True,
    control=True,
    show=True,
    options={
        "maxClusterRadius": 35,
        "spiderfyOnMaxZoom": True,
        "showCoverageOnHover": False,
        "zoomToBoundsOnClick": True
    }
)

marker_cluster.add_to(m)

print("\nMAP COORDINATE VALIDATION")
print("-" * 50)

print("Mapped records:", len(mapped))
print("Missing latitude:", mapped["latitude"].isna().sum())
print("Missing longitude:", mapped["longitude"].isna().sum())

print("\nSample coordinates:")
print(
    mapped[
        ["location_query", "latitude", "longitude", "population_count"]
    ].head(10).to_string(index=False)
)

print("\nLatitude range:")
print(mapped["latitude"].min(), mapped["latitude"].max())

print("\nLongitude range:")
print(mapped["longitude"].min(), mapped["longitude"].max())

# role marker shape

def create_marker_html(role, region, count):

    shape = ROLE_SHAPES.get(
        role,
        "circle"
    )

    color = REGION_COLORS.get(
        region,
        "#808080"
    )

    # Marker size increases with population count.
    radius = min(
        18,
        max(
            6,
            5 + 2.5 * math.log1p(count)
        )
    )

    size = radius * 2

    common_style = (
        f"width:{size}px;"
        f"height:{size}px;"
        f"background-color:{color};"
        "border:2px solid white;"
        "box-shadow:0 1px 4px rgba(0,0,0,0.5);"
    )

    if shape == "circle":

        shape_style = (
            "border-radius:50%;"
        )

    elif shape == "square":

        shape_style = (
            "border-radius:2px;"
        )

    elif shape == "triangle":

        shape_style = (
            "clip-path:polygon(50% 0%, 0% 100%, 100% 100%);"
            "border:none;"
        )

    else:

        shape_style = (
            "border-radius:50%;"
        )

    return f"""
    <div style="
        {common_style}
        {shape_style}
    "></div>
    """


# aggregated markers

for _, row in tqdm(
    mapped.iterrows(),
    total=len(mapped),
    desc="Building map"
):

    role = row["map_role"]
    region = row["map_region"]
    count = int(row["population_count"])

    latitude = float(row["latitude"])
    longitude = float(row["longitude"])

    marker_html = create_marker_html(
        role,
        region,
        count
    )

    city = (
        html.escape(str(row["current_city"]))
        if pd.notna(row["current_city"])
        else "Unknown"
    )

    state = (
        html.escape(str(row["current_state"]))
        if pd.notna(row["current_state"])
        else "Unknown"
    )

    zipcode = (
        html.escape(str(row["current_zip"]))
        if pd.notna(row["current_zip"])
        else "Unknown"
    )

    popup_html = f"""
    <div style="font-family:Arial; width:220px;">

        <h4 style="margin-bottom:8px;">
            Population Location
        </h4>

        <b>Current City:</b> {city}<br>
        <b>State:</b> {state}<br>
        <b>ZIP Code:</b> {zipcode}<br>

        <hr>

        <b>Role:</b> {html.escape(role)}<br>
        <b>Permanent Region:</b> {html.escape(region)}<br>

        <hr>

        <b>Population Count:</b>
        <span style="font-size:18px; font-weight:bold;">
            {count:,}
        </span>

    </div>
    """

    icon = folium.DivIcon(
        html=marker_html,
        icon_size=(40, 40),
        icon_anchor=(20, 20)
    )

    folium.Marker(
        location=[
            latitude,
            longitude
        ],
        icon=icon,
        popup=folium.Popup(
            popup_html,
            max_width=300
        ),
        tooltip=(
            f"{role} | {region} | "
            f"Population: {count:,}"
        )
    ).add_to(marker_cluster)


# map legend

role_legend = """
<div style="
    position: fixed;
    bottom: 25px;
    left: 25px;
    width: 225px;
    background-color: white;
    border: 2px solid #555;
    border-radius: 7px;
    z-index: 9999;
    padding: 12px;
    font-family: Arial;
    font-size: 12px;
    box-shadow: 0 2px 6px rgba(0,0,0,0.3);
">

    <h4 style="margin-top:0;">
        Population Map Legend
    </h4>

    <b>Marker Shape = Role</b>

    <div style="margin-top:8px;">
        <span style="
            display:inline-block;
            width:12px;
            height:12px;
            border-radius:50%;
            background:#777;
            margin-right:8px;
        "></span>
        Student
    </div>

    <div style="margin-top:6px;">
        <span style="
            display:inline-block;
            width:12px;
            height:12px;
            background:#777;
            margin-right:8px;
        "></span>
        Staff/Admin
    </div>

    <div style="margin-top:6px;">
        <span style="
            display:inline-block;
            width:0;
            height:0;
            border-left:7px solid transparent;
            border-right:7px solid transparent;
            border-bottom:13px solid #777;
            margin-right:8px;
        "></span>
        Faculty
    </div>

    <hr>

    <b>Marker Color = Permanent Region</b>

    <div style="margin-top:8px;">
        <span style="color:#2E8B57;">●</span>
        NorCal
    </div>

    <div>
        <span style="color:#2878B5;">●</span>
        San Diego
    </div>

    <div>
        <span style="color:#D62728;">●</span>
        Los Angeles
    </div>

    <div>
        <span style="color:#8A2BE2;">●</span>
        Out of State
    </div>

    <div>
        <span style="color:#F28E2B;">●</span>
        Bay Area
    </div>

    <hr>

    <b>Marker Size = Population Count</b>

    <div style="font-size:11px; color:#555;">
        Larger markers represent more records.
    </div>

</div>
"""

m.get_root().html.add_child(
    folium.Element(role_legend)
)


# control layer

folium.LayerControl(
    collapsed=True
).add_to(m)


# pop usmmary

summary = (
    mapped.groupby(
        ["current_city", "current_state", "current_zip"],
        dropna=False
    )
    .agg(
        total_population=("population_count", "sum"),
        unique_role_region_combinations=(
            "population_count",
            "size"
        )
    )
    .reset_index()
    .sort_values(
        "total_population",
        ascending=False
    )
)

summary.to_csv(
    SUMMARY_PATH,
    index=False
)

folium.Marker(
    location=[40.8021, -124.1637],
    popup="Test Marker - Eureka, California",
    tooltip="Test Marker"
).add_to(m)

# validate

original_records = len(df)

records_represented = mapped[
    "population_count"
].sum()

aggregated_markers = len(grouped)

unmatched_markers = len(unmatched)

print("\n" + "=" * 65)
print("MAP SUCCESSFULLY CREATED")
print("=" * 65)

print(f"Original records: {original_records:,}")
print(f"Aggregated markers: {aggregated_markers:,}")
print(f"Mapped markers: {len(mapped):,}")
print(f"Records represented: {records_represented:,}")
print(f"Unmatched markers: {unmatched_markers:,}")

if records_represented + unmatched["population_count"].sum() != original_records:
    print(
        "\nWARNING: Some records were excluded "
        "or not represented in the aggregation."
    )

print("\nOUTPUT FILES")
print("-" * 65)

print(f"Map: {OUTPUT_MAP}")
print(f"Cache: {CACHE_PATH}")
print(f"Summary: {SUMMARY_PATH}")

print("=" * 65)


# save and open map

m.save(OUTPUT_MAP)

if os.path.exists(OUTPUT_MAP):

    print("\nMap saved successfully.")

    print(
        f"File size: "
        f"{os.path.getsize(OUTPUT_MAP) / 1024:.2f} KB"
    )

    print("\nOpening map in your default browser...")

    webbrowser.open(
        "file:///" + OUTPUT_MAP.replace("\\", "/")
    )

else:

    print("\nERROR: Map file was not created.")



LOADING POPULATION DATA
Loaded records: 7,633
Records with valid ZIP formatting: 7,633

AGGREGATION RESULTS
Original records: 7,633
Aggregated markers: 42
Records represented: 7,633
Unique current locations: 6

Loaded geocoding cache: 12 locations

GEOCODING CURRENT LOCATIONS


Geocoding: 100%|██████████| 6/6 [00:00<00:00, 28992.88it/s]



Successfully geocoded markers: 42
Unmatched aggregated markers: 0

BUILDING MAP

MAP COORDINATE VALIDATION
--------------------------------------------------
Mapped records: 42
Missing latitude: 0
Missing longitude: 0

Sample coordinates:
           location_query  latitude   longitude  population_count
   Arcata, CA, 95521, USA 40.875864 -124.084283               151
   Arcata, CA, 95521, USA 40.875864 -124.084283               147
   Arcata, CA, 95521, USA 40.875864 -124.084283               829
   Arcata, CA, 95521, USA 40.875864 -124.084283               922
   Arcata, CA, 95521, USA 40.875864 -124.084283               644
   Arcata, CA, 95521, USA 40.875864 -124.084283               866
   Arcata, CA, 95521, USA 40.875864 -124.084283               434
Blue Lake, CA, 95525, USA 40.879713 -123.979287                36
Blue Lake, CA, 95525, USA 40.879713 -123.979287                32
Blue Lake, CA, 95525, USA 40.879713 -123.979287                24

Latitude range:
40.5847485 41.067

Building map: 100%|██████████| 42/42 [00:00<00:00, 2699.41it/s]


MAP SUCCESSFULLY CREATED
Original records: 7,633
Aggregated markers: 42
Mapped markers: 42
Records represented: 7,633
Unmatched markers: 0

OUTPUT FILES
-----------------------------------------------------------------
Map: C:\Users\MARE\OneDrive\Documents\data311_project2\data\processed\student_population_role_region_map.html
Cache: C:\Users\MARE\OneDrive\Documents\data311_project2\data\processed\current_location_cache.csv
Summary: C:\Users\MARE\OneDrive\Documents\data311_project2\data\processed\population_location_summary.csv



Map saved successfully.
File size: 90.07 KB

Opening map in your default browser...
